# Type Hints — How FastMCP Knows What Your Tools Expect

FastMCP reads your function’s type hints at runtime to build JSON schemas. Those schemas are what the model receives to understand how to call your tools correctly.

**Topics:** `str · int · float · bool · list · dict · Optional · Annotated · Field · Literal · TypedDict · Pydantic BaseModel`

---
## MCP vs FastMCP

**MCP** is a protocol spec. **FastMCP** is the Python library that implements it. When you write `@mcp.tool()`, FastMCP inspects your type hints, builds a JSON schema, and handles all the protocol communication.

```
Your Python function → FastMCP inspects hints → JSON schema → MCP protocol → model
```

---
## 1. What FastMCP Does With Your Type Hints

| What you write | What the model gets |
|---|---|
| No hint | everything defaults to `string` — even `int` params |
| `city: str, days: int` | correct types in the schema |
| `Annotated[int, Field(description="…")]` | correct type **and** a description |

In [2]:
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("demo")

# Level 1: no hints — FastMCP defaults ALL params to string, even days which should be int
@mcp.tool()
def get_forecast_no_hints(city, days) -> str:
    return f"Forecast for {city} for {days} days"

# Level 2: basic hints — FastMCP now knows city is str and days is int
@mcp.tool()
def get_forecast_typed(city: str, days: int) -> str:
    return f"Forecast for {city} for {days} days"

# Level 3: Annotated + Field — adds descriptions and constraints (covered in Section 5)

# What the model actually receives for each tool
for tool in mcp._tool_manager.list_tools():
    print(f"\n--- {tool.name} ---")
    for name, prop in tool.parameters["properties"].items():
        print(f"  {name}: {prop}")


--- get_forecast_no_hints ---
  city: {'title': 'city', 'type': 'string'}
  days: {'title': 'days', 'type': 'string'}

--- get_forecast_typed ---
  city: {'title': 'City', 'type': 'string'}
  days: {'title': 'Days', 'type': 'integer'}


---
## 2. Basic Types — `str`, `int`, `float`, `bool`

The primitives. Used in almost every tool parameter.

| Python | JSON Schema |
|--------|-------------|
| `str`  | `"string"`  |
| `int`  | `"integer"` |
| `float`| `"number"`  |
| `bool` | `"boolean"` |

---
## 3. Collection Types — `list`, `dict`

Use square brackets to specify what’s **inside** the collection.

- `list[str]` — every element is a string
- `dict[str, int]` — string keys, integer values

In [3]:
# list[T] — every item is type T
cities: list[str] = ["New York", "London", "Tokyo"]
temps: list[float] = [72.1, 65.3, 80.0]

print(cities)
print(temps)

['New York', 'London', 'Tokyo']
[72.1, 65.3, 80.0]


In [4]:
# dict[K, V] — key type K, value type V
scores: dict[str, int] = {"alice": 95, "bob": 87}
prices: dict[str, float] = {"apple": 1.99, "banana": 0.59}

print(scores)
print(prices)

{'alice': 95, 'bob': 87}
{'apple': 1.99, 'banana': 0.59}


---
## 4. `Optional` — Parameters That Might Not Be Provided

**Gotcha:** `Optional[str]` alone does NOT make a parameter optional — it just allows `None`. You must also give it a default.

```python
city: Optional[str]          # still REQUIRED — just allows None
city: Optional[str] = None   # truly optional — not in "required" list
```

In [5]:
from typing import Optional
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("demo")

# Optional[str] without default — still REQUIRED in the schema
@mcp.tool()
def tool_required(city: Optional[str]) -> str:
    return f"City: {city}"

# Optional[str] with default — truly optional, not in "required" list
@mcp.tool()
def tool_optional(city: Optional[str] = None) -> str:
    return f"City: {city or 'unknown'}"

for name in ["tool_required", "tool_optional"]:
    tool = mcp._tool_manager.get_tool(name)
    print(f"\n--- {name} ---")
    print(f"  required: {tool.parameters.get('required', [])}")
    print(f"  city: {tool.parameters['properties']['city']}")


--- tool_required ---
  required: ['city']
  city: {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'title': 'City'}

--- tool_optional ---
  required: []
  city: {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'title': 'City'}


---
## 5. `Annotated` + `Field` — Adding Descriptions

```python
# WRONG — plain string is silently ignored
city: Annotated[str, "The city name"]

# CORRECT — description appears in the schema
city: Annotated[str, Field(description="The city name, e.g. London")]
```

In [6]:
from typing import Annotated
from pydantic import Field
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("demo")

@mcp.tool()
def get_weather(
    city:  Annotated[str, Field(description="The city name, e.g. London")],
    units: Annotated[str, Field(description="Temperature units: celsius or fahrenheit")] = "celsius",
) -> str:
    return f"Weather for {city} in {units}"

tool = mcp._tool_manager.get_tool("get_weather")
for name, prop in tool.parameters["properties"].items():
    print(f"{name}: {prop}")

city: {'description': 'The city name, e.g. London', 'title': 'City', 'type': 'string'}
units: {'default': 'celsius', 'description': 'Temperature units: celsius or fahrenheit', 'title': 'Units', 'type': 'string'}


---
### `Field` Constraints — Restricting What the Model Can Pass

`Field` also accepts validation constraints. FastMCP includes these in the schema, so the model knows the valid range before it even calls your tool.

In [7]:
from typing import Annotated
from pydantic import Field
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("demo")

@mcp.tool()
def get_forecast(
    city:    Annotated[str, Field(description="City name")],
    days:    Annotated[int, Field(description="Days to forecast", ge=1, le=14)],
    unit:    Annotated[str, Field(description="Temperature unit", pattern="^(celsius|fahrenheit)$")] = "celsius",
) -> str:
    return f"Forecast for {city}, {days} days, in {unit}"

tool = mcp._tool_manager.get_tool("get_forecast")
for name, prop in tool.parameters["properties"].items():
    print(f"{name}: {prop}")

city: {'description': 'City name', 'title': 'City', 'type': 'string'}
days: {'description': 'Days to forecast', 'maximum': 14, 'minimum': 1, 'title': 'Days', 'type': 'integer'}
unit: {'default': 'celsius', 'description': 'Temperature unit', 'pattern': '^(celsius|fahrenheit)$', 'title': 'Unit', 'type': 'string'}


---
## 6. `Literal` — Constraining to a Fixed Set of Values

`Literal` produces an `enum` in the schema — the model sees the exact valid choices, not a regex.

```python
# pattern — model must interpret a regex
unit: Annotated[str, Field(pattern="^(celsius|fahrenheit)$")]

# Literal — model sees ["celsius", "fahrenheit"] directly
unit: Literal["celsius", "fahrenheit"]

# combine with Field to add a description
unit: Annotated[Literal["celsius", "fahrenheit"], Field(description="Temperature unit")]
```

In [8]:
from typing import Annotated, Literal
from pydantic import Field
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("demo")

@mcp.tool()
def convert_temperature(
    value: Annotated[float, Field(description="The temperature value to convert")],
    unit:  Annotated[Literal["celsius", "fahrenheit"], Field(description="Unit of the input value")] = "celsius",
) -> str:
    if unit == "celsius":
        return f"{value}°C = {value * 9/5 + 32}°F"
    return f"{value}°F = {(value - 32) * 5/9:.1f}°C"

tool = mcp._tool_manager.get_tool("convert_temperature")
for name, prop in tool.parameters["properties"].items():
    print(f"{name}: {prop}")

# unit produces: {'enum': ['celsius', 'fahrenheit'], 'type': 'string', 'description': '...'}
# The model sees the exact list of valid choices — no regex to interpret

value: {'description': 'The temperature value to convert', 'title': 'Value', 'type': 'number'}
unit: {'default': 'celsius', 'description': 'Unit of the input value', 'enum': ['celsius', 'fahrenheit'], 'title': 'Unit', 'type': 'string'}


---
## 7. `TypedDict` — Typed Dictionaries with Named Fields

A regular dict has no fixed shape. `TypedDict` defines a dict with specific named keys and types — giving you autocomplete, type checking, and a structured schema. Access is still via string keys (`result["city"]`), not dot access.

In [9]:
from typing import TypedDict

class WeatherResult(TypedDict):
    city: str
    temperature: float
    condition: str
    humidity: int

result: WeatherResult = {
    "city": "London",
    "temperature": 65.3,
    "condition": "cloudy",
    "humidity": 80,
}

print(result["city"])
print(result["temperature"])

London
65.3


---
## 8. `Pydantic` — Validation + Auto JSON Schema

FastMCP uses Pydantic under the hood — every tool parameter is validated before your function runs.

| | `TypedDict` | `Pydantic BaseModel` |
|---|---|---|
| Dot access | no | yes |
| Runtime validation | no | **yes** |
| Auto JSON schema | no | **yes** |
| Used by FastMCP | no | **yes** |

In [10]:
from pydantic import BaseModel
from typing import Optional

class WeatherResult(BaseModel):
    city: str
    temperature: float
    condition: str
    humidity: Optional[int] = None

# Pydantic validates on creation — wrong types raise a clear error
result = WeatherResult(city="Tokyo", temperature=72.0, condition="sunny")
print(result)
print(result.city)          # dot access like dataclass

# Pydantic coerces compatible types
result2 = WeatherResult(city="London", temperature="65.3", condition="cloudy")  # str -> float
print(result2.temperature, type(result2.temperature))   # 65.3 <class 'float'>

city='Tokyo' temperature=72.0 condition='sunny' humidity=None
Tokyo
65.3 <class 'float'>


In [11]:
from pydantic import BaseModel, ValidationError

class WeatherInput(BaseModel):
    city: str
    days: int

# Valid input — works fine
print(WeatherInput(city="London", days=7))

# Invalid input — Pydantic raises a clear ValidationError
try:
    WeatherInput(city="London", days="not-a-number")
except ValidationError as e:
    print(e)

city='London' days=7
1 validation error for WeatherInput
days
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='not-a-number', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/int_parsing


In [12]:
from pydantic import BaseModel
from typing import Optional

# Pydantic auto-generates a JSON schema — this is exactly what FastMCP uses
class SearchInput(BaseModel):
    query: str
    max_results: int = 10
    language: Optional[str] = None

schema = SearchInput.model_json_schema()
print("required:", schema["required"])
for name, prop in schema["properties"].items():
    print(f"  {name}: {prop}")

required: ['query']
  query: {'title': 'Query', 'type': 'string'}
  max_results: {'default': 10, 'title': 'Max Results', 'type': 'integer'}
  language: {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'title': 'Language'}


---
### Using `BaseModel` as a FastMCP Tool Parameter

You can use a `BaseModel` directly as a tool parameter type — FastMCP embeds the model’s schema in the tool definition using a `$ref` reference.

In [13]:
from pydantic import BaseModel
from typing import Optional
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("demo")

class SearchInput(BaseModel):
    query: str
    max_results: int = 10
    language: Optional[str] = None

class SearchResult(BaseModel):
    title: str
    url: str
    score: float

# BaseModel as a tool parameter — FastMCP embeds the schema via $ref
# async/await is covered in 04_async_await/ — plain def used here
@mcp.tool()
def search(input: SearchInput) -> list[SearchResult]:
    """Run a search with the given parameters."""
    lang = input.language or "en"
    return [
        SearchResult(title=f"Result {i} ({lang})", url=f"https://example.com/{i}", score=round(0.9 - i*0.1, 1))
        for i in range(min(input.max_results, 3))
    ]

# What the model sees for the input parameter
tool = mcp._tool_manager.get_tool("search")
params = tool.parameters
print("required:", params.get("required", []))
print("input:", params["properties"]["input"])
print()
print("SearchInput fields:")
for name, prop in params["$defs"]["SearchInput"]["properties"].items():
    print(f"  {name}: {prop}")

required: ['input']
input: {'$ref': '#/$defs/SearchInput'}

SearchInput fields:
  query: {'title': 'Query', 'type': 'string'}
  max_results: {'default': 10, 'title': 'Max Results', 'type': 'integer'}
  language: {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'title': 'Language'}
